# GPT-5.4-mini(API)를 활용한 뉴스 3줄 요약

프롬프트 참고문헌: Ali, S., & Wongsawat, A. (2024). A Multi-Stage Prompt Framework for High-Quality News Summarization with Large Language Models.

---

## 실행 환경 요약

- 모델: `gpt-5.4-mini` / API: OpenAI **Responses API** (`client.responses.create`) + JSON Schema 구조화 출력(`strict: true`)
- `reasoning_effort="none"` (분류·추출·요약 과제에 적합). `temperature`는 `reasoning_effort="none"`일 때만 지정할 수 있습니다.
- 이전 버전: [GPT-4.1 버전 저장소](https://github.com/HanJunLeeKR/News_Content_Analysis_with_GPT-4.1) (Chat Completions + `json_object`)
- 두 버전의 차이는 저장소 `README.md`의 「GPT-4.1 버전과의 차이」를 참고하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 패키지 설치 (Colab/Jupyter용). 로컬 환경에서는 저장소의 requirements.txt 사용:
#   pip install -r requirements.txt
# ═══════════════════════════════════════════════════════════════════════
%pip install --upgrade openai pandas tqdm nest-asyncio


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 뉴스 3줄 요약 분석기 (News Summary Generator)
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
from openai import AsyncOpenAI
import asyncio
import json
import os
import logging
import time
from typing import Optional, Tuple, Dict
import re
from tqdm import tqdm

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1. 프롬프트 템플릿
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """Summarize the following Korean news article in exactly 3 sentences.

[NEWS ARTICLE]:
{text}

Rules:
1. Identify 5W1H (who, what, when, where, why, how) from the article.
2. Combine the identified elements into a logically coherent narrative.
3. Maintain a neutral tone.
4. Write in formal Korean (문어체).
5. Write exactly 3 sentences.
6. Separate each sentence with a period (마침표).
7. Do not use honorifics (경어 사용 금지).

Output ONLY valid JSON. Do not add fields not in the schema."""

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. JSON 파서 (EnhancedJSONParser — 단순 json.loads 대비 오류 복원력 강화)
# ═══════════════════════════════════════════════════════════════════════
class EnhancedJSONParser:
    """강화된 JSON 파싱 클래스"""

    @staticmethod
    def extract_json_robust(response_text: str) -> Dict:
        if not response_text or not response_text.strip():
            return {'summary': '', 'error': '빈 응답'}
        response_text = response_text.strip()

        try:
            if response_text.startswith('{') and response_text.endswith('}'):
                return json.loads(response_text)
        except json.JSONDecodeError:
            pass

        for pattern in [r'```json\s*\n?(.*?)\n?```', r'```\s*\n?(.*?)\n?```']:
            match = re.search(pattern, response_text, re.DOTALL | re.IGNORECASE)
            if match:
                try:
                    return json.loads(match.group(1).strip())
                except json.JSONDecodeError:
                    continue

        for pattern in [r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', r'\{(?:[^{}]|{[^{}]*})*\}']:
            for match in re.findall(pattern, response_text):
                try:
                    result = json.loads(match)
                    if 'summary' in result:
                        return result
                except json.JSONDecodeError:
                    continue

        for p in [r'"summary"\s*:\s*"([^"]*(?:\\.[^"]*)*)"']:
            m = re.search(p, response_text, re.IGNORECASE | re.DOTALL)
            if m:
                return {'summary': m.group(1).strip()}

        return {'summary': '', 'error': '파싱 실패'}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3. 뉴스 요약 분석기 (GPT-5.4-mini 버전)
# ═══════════════════════════════════════════════════════════════════════
class NewsSummaryAnalyzer:
    """뉴스 3줄 요약 분석기 (GPT-5.4-mini용)"""

    def __init__(self, api_key, model, output_dir='.',
                 concurrent_requests=2, rate_limit_delay=1.0, batch_size=5,
                 reasoning_effort='none', text_verbosity='medium',
                 temperature: Optional[float] = None,
                 debug=False):
        # ── GPT-5.4-mini: temperature는 reasoning_effort='none'일 때만 유효 ──
        if temperature is not None and reasoning_effort != 'none':
            raise ValueError(
                f"temperature is only valid when reasoning_effort='none'. "
                f"Got reasoning_effort='{reasoning_effort}'."
            )
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.output_dir = output_dir
        self.concurrent_requests = concurrent_requests
        self.rate_limit_delay = rate_limit_delay
        self.batch_size = batch_size
        self.reasoning_effort = reasoning_effort
        self.text_verbosity = text_verbosity
        self.temperature = temperature
        self.debug = debug
        self.total_cost = 0.0
        self.semaphore = asyncio.Semaphore(concurrent_requests)
        self.parsing_successes = 0
        self.parsing_failures = 0
        logging.basicConfig(level=logging.WARNING, format='%(message)s')
        self.logger = logging.getLogger(__name__)
        os.makedirs(self.output_dir, exist_ok=True)

    def combine_title_text(self, title: str, text: str) -> str:
        if pd.isna(title): title = ""
        if pd.isna(text): text = ""
        title = str(title).strip()
        text = str(text).strip()
        combined = f"{title}\n\n{text}" if title and text else (title or text)
        return combined[:10000]

    async def analyze_single_article(self, title: str, text: str) -> Tuple[str, float]:
        async with self.semaphore:
            try:
                combined_text = self.combine_title_text(title, text)
                if not combined_text:
                    return "요약 실패: 빈 텍스트", 0.0

                prompt = PROMPT_TEMPLATE.format(text=combined_text)

                # ═══════════════════════════════════════════════════
                # GPT-5.4-mini API 호출
                # reasoning_effort options: none | low | medium | high | xhigh
                # temperature: only valid when reasoning_effort='none'
                # ═══════════════════════════════════════════════════
                api_params = dict(
                    model=self.model,
                    input=[{"role": "user", "content": prompt}],
                    max_output_tokens=512,
                    reasoning={"effort": self.reasoning_effort},
                    text={
                        "verbosity": self.text_verbosity,
                        "format": {
                            "type": "json_schema",
                            "name": "news_summary",
                            "strict": True,
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "summary": {
                                        "type": "string",
                                        "description": "3-sentence Korean news summary in formal style (문어체), neutral tone, no honorifics"
                                    }
                                },
                                "required": ["summary"],
                                "additionalProperties": False
                            }
                        }
                    }
                )
                if self.temperature is not None:
                    api_params["temperature"] = self.temperature

                response = await self.client.responses.create(**api_params)

                if hasattr(response, 'output_text') and response.output_text:
                    response_text = response.output_text
                    if self.debug:
                        print(f"[DEBUG] output_text: {response_text[:200]}...")
                else:
                    response_text = ""
                    for item in response.output:
                        if item.type == "message":
                            for content in item.content:
                                if hasattr(content, 'text'):
                                    response_text += content.text
                    if self.debug:
                        print(f"[DEBUG] extracted text: {response_text[:200]}...")

                response_text = response_text.strip()
                if not response_text:
                    return "요약 실패: 빈 응답", 0.0

                result  = EnhancedJSONParser.extract_json_robust(response_text)
                summary = result.get('summary', '')

                if summary and 'error' not in result:
                    self.parsing_successes += 1
                else:
                    self.parsing_failures += 1
                    return "요약 실패: summary 필드 없음", 0.0

                input_tokens  = response.usage.input_tokens if hasattr(response.usage, 'input_tokens') else 0
                output_tokens = response.usage.output_tokens if hasattr(response.usage, 'output_tokens') else 0
                estimated_cost = (input_tokens * PRICE_INPUT_PER_1M / 1_000_000) + (output_tokens * PRICE_OUTPUT_PER_1M / 1_000_000)
                self.total_cost += estimated_cost

                return summary, estimated_cost

            except Exception as e:
                if self.debug:
                    print(f"[DEBUG] Exception: {type(e).__name__}: {str(e)}")
                self.parsing_failures += 1
                return f"오류: {str(e)[:50]}", 0.0
            finally:
                await asyncio.sleep(self.rate_limit_delay)

    async def process_batch(self, batch_df) -> Tuple[list, list]:
        tasks = [
            self.analyze_single_article(row['title'], row['text'])
            for _, row in batch_df.iterrows()
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        summaries, costs = [], []
        for result in results:
            if isinstance(result, Exception):
                summaries.append(f"배치 오류: {str(result)[:50]}")
                costs.append(0.0)
                self.parsing_failures += 1
            else:
                summaries.append(result[0])
                costs.append(result[1])
        return summaries, costs

    async def analyze_dataframe(self, df, output_filename="news_summary_results.csv"):
        output_path = os.path.join(self.output_dir, output_filename)
        all_summaries, all_costs = [], []
        start_time = time.time()
        pbar = tqdm(total=len(df), desc="뉴스 요약 중", unit="건")

        for i in range(0, len(df), self.batch_size):
            batch_df = df.iloc[i:min(i + self.batch_size, len(df))]
            try:
                summaries, costs = await self.process_batch(batch_df)
                all_summaries.extend(summaries)
                all_costs.extend(costs)
                pbar.update(len(batch_df))

                batch_num = i // self.batch_size + 1
                if batch_num % 5 == 0:
                    n = len(df)
                    temp_df = df.copy()
                    temp_df['요약'] = all_summaries + [''] * (n - len(all_summaries))
                    temp_df['비용'] = all_costs + [0.0] * (n - len(all_costs))
                    temp_df.to_csv(output_path, encoding='utf-8-sig', index=False)
            except Exception as e:
                sz = len(batch_df)
                all_summaries.extend([f"배치 오류: {str(e)[:30]}"] * sz)
                all_costs.extend([0.0] * sz)
                self.parsing_failures += sz
                pbar.update(sz)

        pbar.close()
        result_df = df.copy()
        result_df['요약'] = all_summaries
        result_df['비용'] = all_costs
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)

        elapsed_time = time.time() - start_time
        self._print_completion_message(result_df, elapsed_time, output_path)
        return result_df

    def _print_completion_message(self, result_df, elapsed_time, output_path):
        total_count   = len(result_df)
        success_count = self.parsing_successes
        fail_count    = self.parsing_failures
        print(f"\n{'='*60}")
        print(f"✅ 뉴스 3줄 요약 완료! (GPT-5.4-mini)")
        print(f"{'='*60}")
        print(f"📊 총 {total_count:,}건 뉴스 요약 완료 (소요시간: {elapsed_time/60:.1f}분)")
        print(f"💰 총 비용: ${self.total_cost:.4f}")
        print(f"📁 결과 파일: {output_path}")
        print(f"\n📈 요약 성공률:")
        if total_count > 0:
            success_rate = (success_count / total_count) * 100
            print(f"   • 성공: {success_count}건 ({success_rate:.1f}%)")
            print(f"   • 실패: {fail_count}건 ({(fail_count/total_count)*100:.1f}%)")
        print(f"{'='*60}\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. 설정 및 실행
# ═══════════════════════════════════════════════════════════════════════
# ──────────────────────────────────────────────────────────────────────
# 여기서 설정을 변경하세요
# ──────────────────────────────────────────────────────────────────────
MODEL = "gpt-5.4-mini"          # GPT-5.4-mini
API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")  # 환경변수 사용 권장. 키를 코드에 직접 적은 채 커밋하지 마세요.
INPUT_CSV_PATH  = "Input CSV file path here"
OUTPUT_CSV_PATH = "Output CSV file path here"

# GPT-5.4-mini 전용 설정
# reasoning_effort options: none | low | medium | high | xhigh
# ※ summarization 태스크에는 'none' 권장
REASONING_EFFORT = "none"
TEXT_VERBOSITY   = "medium"     # low | medium | high
# temperature: reasoning_effort='none'일 때만 유효 (None이면 비활성화)
TEMPERATURE      = None
# ※ GPT-4.1 버전은 temperature=0.0 고정이었습니다. 동일한 재현성이 필요하면 0.0으로 설정하세요(reasoning_effort="none"일 때만 유효).
DEBUG_MODE       = False

# 비용 추정용 단가 (USD / 1M tokens). OpenAI 공식 가격표 기준 gpt-5.4-mini 입력 $0.75, 출력 $4.50 (2026-11 확인). 가격이 바뀌면 수정하세요.
PRICE_INPUT_PER_1M  = 0.75
PRICE_OUTPUT_PER_1M = 4.50
# ──────────────────────────────────────────────────────────────────────

async def main():
    try:
        df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8-sig')
        if 'title' not in df.columns or 'text' not in df.columns:
            print("❌ title, text 컬럼 필요")
            return

        output_dir  = os.path.dirname(OUTPUT_CSV_PATH) or '.'
        output_file = os.path.basename(OUTPUT_CSV_PATH)

        analyzer = NewsSummaryAnalyzer(
            api_key=API_KEY,
            model=MODEL,
            output_dir=output_dir,
            reasoning_effort=REASONING_EFFORT,
            text_verbosity=TEXT_VERBOSITY,
            temperature=TEMPERATURE,
            debug=DEBUG_MODE
        )
        await analyzer.analyze_dataframe(df, output_file)

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없음: {INPUT_CSV_PATH}")
    except Exception as e:
        print(f"❌ 오류: {str(e)}")
        import traceback; traceback.print_exc()

def run_analysis():
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass
    asyncio.run(main())

if __name__ == "__main__":
    run_analysis()